# Simple Microgrid Kaggle Notebook

用于在 Kaggle 上运行 `microgrid_sim` 的 DRL、Oracle 和 heuristic-lite 对照，并导出论文可用结果工件。

默认主链路：
- `scripts/analysis/short_cross_fidelity_probe.py`
- 可选：`scripts/analysis/full_year_oracle_compare.py`
- 可选：`scripts/analysis/full_year_heuristic_lite_compare.py`

默认已经切到严肃长跑配置：`IEEE33 + SAC + 300k + 365d held-out eval`。

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys

print('Python:', sys.version)
print('CWD:', Path.cwd())
print('Has /kaggle/input:', Path('/kaggle/input').exists())
print('Has /kaggle/working:', Path('/kaggle/working').exists())
print('CPU count:', os.cpu_count())


## 0) 参数区

只改这一格。Notebook 后续所有命令都会从这里读取配置。

In [ ]:
from dataclasses import dataclass


@dataclass
class RunCfg:
    # Kaggle 代码数据集目录。
    # 推荐填仓库根目录，而不是 `src/microgrid_sim` 这种子目录。
    # 当前 notebook 也支持你误填到子目录时自动向上寻找 pyproject.toml。
    kaggle_codeset_dir: str = '/kaggle/input/datasets/tttwwwjjj/microgrid-code'

    # Kaggle 数据数据集目录。
    # 这里建议直接指向 `.../microgrid-data/data/processed`。
    # notebook 会把它复制到 `work_root/data/processed`，从而兼容仓库默认数据解析逻辑。
    kaggle_data_processed_dir: str = '/kaggle/input/datasets/tttwwwjjj/microgrid-data/data/processed'

    # 可选：上一次运行导出的结果目录，用于把旧的 results/ 合并进当前工作目录。
    # 不续跑时保持空字符串即可。
    kaggle_prev_output_dir: str = ''

    # Kaggle 可写工作目录。代码会被复制到这里，再做 editable install。
    work_root: str = '/kaggle/working/microgrid_sim'

    # 是否把 /kaggle/input 中的代码数据集复制到 work_root。
    # 默认建议保持 True，避免直接在只读目录上运行。
    sync_code_to_workdir: bool = True

    # 最小依赖集合。这里显式安装 RL、潮流和结果分析所需包。
    install_packages: tuple[str, ...] = (
        'numpy',
        'pandas',
        'scipy',
        'matplotlib',
        'gymnasium',
        'stable-baselines3',
        'sb3-contrib',
        'pandapower',
        'tensorboard',
        'tqdm',
    )

    # 实验名会直接用于 results/<experiment_name>/ 输出目录与压缩包命名。
    # 默认切到 serious 年度协议，方便直接做论文主结果候选运行。
    experiment_name: str = 'kaggle_ieee33_sac_300k_train2023_eval2024_365d'

    # 主实验电网。默认使用 IEEE33，因为它当前比 CIGRE 更适合作为 DRL 主实验台。
    case: str = 'ieee33'

    # 工况。默认 network_stress，用于放大储能调度价值与约束压力。
    regime: str = 'network_stress'

    # 奖励配置。默认使用当前最适合库存闭环和电池可行性学习的 paper_balanced。
    reward_profile: str = 'paper_balanced'

    # DRL 算法。默认用 SAC，不是因为它一定最终最优，而是因为当前本地短测里
    # 它比 PPO 更不容易立刻掉进低 SOC 边界吸引子，更适合先冲年度长跑。
    agent: str = 'sac'

    # 训练电池模型与测试电池模型。
    # 默认先用 simple->simple 做主结果，这条线最稳、最容易先拿到收敛证据。
    # 若要做跨保真部署失配，可把 test_model 改成 thevenin，或把 train_model 改成 simple+thevenin。
    train_model: str = 'simple'
    test_model: str = 'simple'

    # 训练步数。默认 300k，属于 Kaggle 上偏重的 serious budget。
    # 如果只是 smoke/scout，可先降到 5k / 20k。
    train_steps: int = 300000

    # 环境基准长度。这里保持 365 天，使训练与评估都在全年尺度下定义。
    days: int = 365

    # 年度切分协议：2023 训练、2024 评估。
    # 这是当前仓库中最适合论文叙述的 held-out generalization 设置。
    train_year: int = 2023
    eval_year: int = 2024

    # 训练时每个 episode 的采样窗口长度。
    # 30 天是当前推荐折中：既能保留库存闭环，又不至于让每个 episode 过长。
    train_episode_days: int = 30

    # 正式评估窗口长度。默认直接评估 365 天。
    # 若只是试运行，可改成 7 或 30。
    eval_days: int = 365

    # 是否忽略 eval_steps，直接按 eval_days/full-horizon 滚完整个评估窗口。
    # 全年结论必须保持 True。
    eval_full_horizon: bool = True

    # 随机种子。正式论文建议至少补多个 seed，但单次 Kaggle 长跑先从 seed42 开始。
    seed: int = 42

    # 设备。默认 auto，让 SB3 优先吃 Kaggle GPU；若环境不支持会自动回退。
    device: str = 'auto'

    # 连续动作正则：
    # smoothing 用来减少抖动，max_delta 限制步间突变，rate_penalty 抑制高频来回切换。
    # 这是当前最推荐的一组稳定化参数。
    action_smoothing_coef: float = 0.5
    action_max_delta: float = 0.1
    action_rate_penalty: float = 0.05

    # 是否启用 SOC 可行域感知裁剪。建议保持 True。
    battery_feasibility_aware: bool = True

    # 对不可行动作施加的负奖励系数。
    # 数值越负，越会惩罚“想放电但其实已经没电/想充电但已经满了”的无效请求。
    battery_infeasible_penalty: float = -1.0

    # 是否把正向 battery action 缩放到与 charge/discharge 可用范围更对称的区间。
    # 建议保持 True，可减少单边动作偏置。
    symmetric_battery_action: bool = True

    # 是否顺带跑全年 Oracle。
    # Oracle 很适合作为论文上界参考，默认打开。
    run_oracle: bool = True

    # 是否顺带跑全年 heuristic-lite。
    # 这是全年快速次优对照，默认关闭；若你希望一次性拿到算法差距分析，可切成 True。
    run_heuristic_lite: bool = False


cfg = RunCfg()
cfg

## 1) 同步代码到 `/kaggle/working`

In [ ]:
from pathlib import Path
import shutil


def find_codeset(preferred: str) -> Path:
    preferred_path = Path(preferred)
    if preferred_path.exists():
        for candidate in [preferred_path, *preferred_path.parents]:
            if (candidate / 'pyproject.toml').exists() and (candidate / 'src').exists():
                return candidate
    candidates = []
    root = Path('/kaggle/input')
    for path in root.rglob('*'):
        if path.is_dir() and (path / 'pyproject.toml').exists() and (path / 'src').exists():
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError('No Kaggle input dataset containing pyproject.toml and src/ was found.')
    return candidates[0]


def find_processed_data_dir(preferred: str) -> Path | None:
    if not preferred:
        return None
    preferred_path = Path(preferred)
    if preferred_path.exists():
        if (preferred_path / 'network_15min').exists():
            return preferred_path
        if preferred_path.name == 'network_15min':
            return preferred_path.parent
    root = Path('/kaggle/input')
    for path in root.rglob('processed'):
        if (path / 'network_15min').exists():
            return path
    return None


source_repo = find_codeset(cfg.kaggle_codeset_dir)
processed_data_dir = find_processed_data_dir(cfg.kaggle_data_processed_dir)
work_root = Path(cfg.work_root)
if cfg.sync_code_to_workdir:
    if work_root.exists():
        shutil.rmtree(work_root)
    shutil.copytree(source_repo, work_root)

if processed_data_dir is not None:
    target_processed_dir = work_root / 'data' / 'processed'
    target_processed_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(processed_data_dir, target_processed_dir, dirs_exist_ok=True)

if cfg.kaggle_prev_output_dir and Path(cfg.kaggle_prev_output_dir).exists():
    prev_dir = Path(cfg.kaggle_prev_output_dir)
    if (prev_dir / 'results').exists():
        shutil.copytree(prev_dir / 'results', work_root / 'results', dirs_exist_ok=True)

print('source_repo =', source_repo)
print('processed_data_dir =', processed_data_dir)
print('work_root =', work_root)


## 2) 安装依赖并安装仓库

In [ ]:
import subprocess
import sys


subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *cfg.install_packages], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(work_root)], check=True)
print('editable install done')


## 3) 运行 DRL 主实验

In [ ]:
from pathlib import Path
import subprocess
import sys


output_dir = work_root / 'results' / cfg.experiment_name
output_dir.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(work_root / 'scripts' / 'analysis' / 'short_cross_fidelity_probe.py'),
    '--cases', cfg.case,
    '--regimes', cfg.regime,
    '--reward-profile', cfg.reward_profile,
    '--agent', cfg.agent,
    '--train-models', cfg.train_model,
    '--test-models', cfg.test_model,
    '--train-steps', str(cfg.train_steps),
    '--eval-steps', '0',
    '--days', str(cfg.days),
    '--train-year', str(cfg.train_year),
    '--train-episode-days', str(cfg.train_episode_days),
    '--train-random-start-within-year',
    '--eval-year', str(cfg.eval_year),
    '--eval-days', str(cfg.eval_days),
    '--seed', str(cfg.seed),
    '--device', cfg.device,
    '--action-smoothing-coef', str(cfg.action_smoothing_coef),
    '--action-max-delta', str(cfg.action_max_delta),
    '--action-rate-penalty', str(cfg.action_rate_penalty),
    '--output-dir', str(output_dir),
]

if cfg.eval_full_horizon:
    cmd.append('--eval-full-horizon')
if cfg.battery_feasibility_aware:
    cmd.append('--battery-feasibility-aware')
    cmd.extend(['--battery-infeasible-penalty', str(cfg.battery_infeasible_penalty)])
if cfg.symmetric_battery_action:
    cmd.append('--symmetric-battery-action')

print('Running command:')
print(' '.join(cmd))
completed = subprocess.run(cmd, cwd=work_root, text=True, capture_output=True, check=True)
print(completed.stdout)
if completed.stderr.strip():
    print('STDERR:')
    print(completed.stderr)


## 4) 读取结果摘要

In [ ]:
import pandas as pd


summary_csv = output_dir / 'summary.csv'
summary_df = pd.read_csv(summary_csv)
display_cols = [
    'case',
    'regime',
    'agent',
    'train_model',
    'test_model',
    'train_steps',
    'steps',
    'final_cumulative_cost',
    'final_cumulative_objective_cost',
    'final_soc',
    'total_terminal_soc_penalty',
    'soc_lower_dwell_fraction',
    'soc_upper_dwell_fraction',
    'infeasible_action_dwell_fraction',
    'mean_battery_action_infeasible_gap',
]
summary_df[display_cols]


## 5) 可选：运行全年 Oracle

In [ ]:
if cfg.run_oracle:
    oracle_dir = work_root / 'results' / f'{cfg.experiment_name}_oracle'
    oracle_cmd = [
        sys.executable,
        str(work_root / 'scripts' / 'analysis' / 'full_year_oracle_compare.py'),
        '--cases', cfg.case,
        '--regimes', cfg.regime,
        '--reward-profile', cfg.reward_profile,
        '--battery-model', 'simple',
        '--days', '365',
        '--seed', str(cfg.seed),
        '--efficiency-model', 'realistic',
        '--output-dir', str(oracle_dir),
    ]
    print(' '.join(oracle_cmd))
    oracle_completed = subprocess.run(oracle_cmd, cwd=work_root, text=True, capture_output=True, check=True)
    print(oracle_completed.stdout)
else:
    print('skip oracle')


## 6) 可选：运行全年 heuristic-lite

In [ ]:
if cfg.run_heuristic_lite:
    heuristic_dir = work_root / 'results' / f'{cfg.experiment_name}_heuristic_lite'
    oracle_summary_csv = work_root / 'results' / f'{cfg.experiment_name}_oracle' / 'protocol_summary.csv'
    heuristic_cmd = [
        sys.executable,
        str(work_root / 'scripts' / 'analysis' / 'full_year_heuristic_lite_compare.py'),
        '--cases', cfg.case,
        '--regimes', cfg.regime,
        '--reward-profile', cfg.reward_profile,
        '--battery-model', 'simple',
        '--days', '365',
        '--seed', str(cfg.seed),
        '--baselines', 'none,heuristic_blended,heuristic_selector_lite',
        '--selector-window-days', '7',
        '--selector-stride-days', '1',
        '--evaluation-mode', 'surrogate',
        '--oracle-summary-csv', str(oracle_summary_csv),
        '--output-dir', str(heuristic_dir),
    ]
    print(' '.join(heuristic_cmd))
    heuristic_completed = subprocess.run(heuristic_cmd, cwd=work_root, text=True, capture_output=True, check=True)
    print(heuristic_completed.stdout)
else:
    print('skip heuristic-lite')


## 7) 打包输出

In [ ]:
import shutil


archive_base = work_root / 'results' / cfg.experiment_name
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=output_dir)
print('archive_path =', archive_path)
